# 06 — Universe Live (flex_params sekarang) + BTC Ikut Ranking, Sebelum Jadi Endpoint

Keputusan user (1 Okt 2026): rekomendasi **harian**, urutan boleh berubah tiap hari ("hari ini BTC no 1, besok ETH").
Sebelum endpoint dibuat, dua hal berbeda dari riset 01–05 harus diuji:

1. **Universe**: `flex_params` Large/Mid di DB sekarang cuma **23 simbol** (riset memakai snapshot 40 coin).
   Minus USDC & PAXG = **21 coin**. Skor = persentil antar coin, jadi "top-10 dari 21" ≠ "top-10 dari 40".
2. **BTC ikut kandidat** (riset sebelumnya: BTC dikeluarkan, hanya jadi filter pasar).

Exit = **hold 14 hari + stop −25%** (notebook 05). Semua fungsi dari `src/strategy/momentum.py` — kode yang sama
yang dipakai API, jadi notebook ini juga **cek paritas** kode produksi.

Tidak ada parameter yang dipilih di sini (tidak menambah percobaan) — ini pengukuran ulang untuk angka yang akan
ditampilkan ke user. Periode setelah 30 Juni 2026 (akhir data riset) adalah **out-of-sample murni** dan akan
dipantau lewat endpoint riwayat (paper trading).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from data_source.fear_greed import load_fear_greed_index
from src.strategy import momentum as mo
from src.strategy.allocation import market_filter

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
BTC = "BTCUSDT"
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))
RESEARCH40 = [s for s in ALL_SYMS if s not in mo.EXCLUDED]
# Snapshot flex_params Large/Mid aktif per 1 Okt 2026 (query read-only di sesi riset)
FLEX_LIVE = ["AAVEUSDT", "ADAUSDT", "ALGOUSDT", "AVAXUSDT", "BCHUSDT", "BNBUSDT", "BTCUSDT", "DOGEUSDT", "DOTUSDT",
             "ETCUSDT", "ETHUSDT", "ICPUSDT", "LINKUSDT", "LTCUSDT", "PAXGUSDT", "QNTUSDT", "SKYUSDT", "SOLUSDT",
             "TRXUSDT", "UNIUSDT", "USDCUSDT", "XLMUSDT", "XRPUSDT"]
LIVE21 = [s for s in FLEX_LIVE if s not in mo.EXCLUDED]

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in RESEARCH40}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
px = {f: pd.DataFrame({s: raw[s][f].reindex(dates) for s in RESEARCH40}) for f in ("open", "low", "close")}
fgi = load_fear_greed_index()["fng_value"].reindex(dates).ffill()
market_ok = market_filter(px["close"][BTC], fgi)["risk_on"].fillna(False).to_numpy(bool)
print(f"Riset: {len(RESEARCH40)} coin | Live: {len(LIVE21)} coin -> {LIVE21}")

Riset: 40 coin | Live: 21 coin -> ['AAVEUSDT', 'ADAUSDT', 'ALGOUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETCUSDT', 'ETHUSDT', 'ICPUSDT', 'LINKUSDT', 'LTCUSDT', 'QNTUSDT', 'SKYUSDT', 'SOLUSDT', 'TRXUSDT', 'UNIUSDT', 'XLMUSDT', 'XRPUSDT']


## Mesin: semua sinyal top-10 harian (risk-on & risk-off) untuk satu konfigurasi

In [2]:
def run(universe, btc_candidate):
    close = px["close"][universe]
    score, _, _ = mo.momentum_score(close)
    cands = universe if btc_candidate else [c for c in universe if c != BTC]
    O, L, C = (px[f][universe].to_numpy(float) for f in ("open", "low", "close"))
    col = {s: k for k, s in enumerate(universe)}
    rows = []
    for i in range(len(dates) - 1):
        top = mo.rank_day(score.iloc[i][cands])
        if len(top) < mo.MAX_RECOMMENDATIONS:
            continue
        for rank, sym in enumerate(top, start=1):
            k = col[sym]
            t = mo.simulate_trade(O[:, k], L[:, k], C[:, k], i)
            if t is None or t["status"] == "BERJALAN":
                continue
            rows.append(dict(date=dates[i], coin=sym, rank=rank, group=mo.group_of(rank),
                             market_ok=bool(market_ok[i]), status=t["status"], ret=t["ret"]))
    d = pd.DataFrame(rows)
    d["year"] = d.date.dt.year
    return d

CONFIGS = {
    "riset 40, tanpa BTC": (RESEARCH40, False),
    "live 21, tanpa BTC": (LIVE21, False),
    "live 21, BTC ikut": (LIVE21, True),
}
SIG = {name: run(*cfg) for name, cfg in CONFIGS.items()}
for name, d in SIG.items():
    print(f"{name:22s}: {len(d):,} sinyal selesai")

riset 40, tanpa BTC   : 19,801 sinyal selesai
live 21, tanpa BTC    : 19,792 sinyal selesai
live 21, BTC ikut     : 19,792 sinyal selesai


## 0. Cek paritas kode produksi

`riset 40, tanpa BTC`, top-5 risk-on, harus sama dengan notebook 05 varian **E2 hold 14 hari**:
dev 2021–23 n=2785, ekspektasi +9,14%; 2024 n=1565, +4,33%; 2025–26 n=1290, −1,51%.

In [3]:
d = SIG["riset 40, tanpa BTC"]
d5 = d[(d["rank"] <= 5) & d.market_ok]
for name, (a, b) in {"dev 2021-23": ("2021-01-01", "2023-12-31"), "2024": ("2024-01-01", "2024-12-31"),
                     "2025-26": ("2025-01-01", "2026-06-29")}.items():
    x = d5[(d5.date >= a) & (d5.date <= b)]
    print(f"{name:12s} n={len(x):5d} win_rate={(x.ret > 0).mean():.1%} ekspektasi={x.ret.mean():+.2%}")

dev 2021-23  n= 2785 win_rate=48.2% ekspektasi=+9.08%
2024         n= 1565 win_rate=46.1% ekspektasi=+4.38%
2025-26      n= 1290 win_rate=43.7% ekspektasi=-1.73%


## 1. Statistik per kelompok, per konfigurasi (2024-01 → 2026-06 dan dev 2021–23)

In [4]:
def summ(x):
    if len(x) < 20:
        return dict(n=len(x), win_rate=np.nan, rata2_untung=np.nan, rata2_rugi=np.nan, ekspektasi=np.nan, kena_stop=np.nan)
    w = x.ret > 0
    return dict(n=len(x), win_rate=w.mean(), rata2_untung=x.ret[w].mean(), rata2_rugi=x.ret[~w].mean(),
                ekspektasi=x.ret.mean(), kena_stop=(x.status == "STOP_KERAS").mean())

def fmt(df):
    out = df.copy()
    for c in ("win_rate", "kena_stop"):
        out[c] = df[c].map(lambda v: f"{v:.1%}" if pd.notna(v) else "-")
    for c in ("rata2_untung", "rata2_rugi", "ekspektasi"):
        out[c] = df[c].map(lambda v: f"{v:+.2%}" if pd.notna(v) else "-")
    return out

rows = []
for name, d in SIG.items():
    for pname, (a, b) in {"dev 2021-23": ("2021-01-01", "2023-12-31"), "uji 2024-26": ("2024-01-01", "2026-06-29")}.items():
        p = d[(d.date >= a) & (d.date <= b)]
        for g in ("UTAMA", "PELENGKAP"):
            rows.append(dict(konfigurasi=name, periode=pname, pasar="risk-on", kelompok=g,
                             **summ(p[p.market_ok & (p.group == g)])))
        rows.append(dict(konfigurasi=name, periode=pname, pasar="risk-off", kelompok="UTAMA",
                         **summ(p[~p.market_ok & (p.group == "UTAMA")])))
T1 = pd.DataFrame(rows).set_index(["konfigurasi", "periode", "pasar", "kelompok"])
fmt(T1)

n win_rate rata2_untung rata2_rugi ekspektasi kena_stop
konfigurasi         periode     pasar    kelompok                                                             
riset 40, tanpa BTC dev 2021-23 risk-on  UTAMA      2785    48.2%      +33.10%    -13.22%     +9.08%     15.5%
                                         PELENGKAP  2785    50.1%      +28.00%    -12.86%     +7.59%     14.6%
                                risk-off UTAMA      2620    42.1%      +17.06%    -14.41%     -1.16%     19.8%
                    uji 2024-26 risk-on  UTAMA      2855    45.0%      +19.98%    -13.42%     +1.62%     16.9%
                                         PELENGKAP  2855    43.5%      +19.42%    -12.70%     +1.28%     14.5%
                                risk-off UTAMA      1644    43.7%      +12.09%    -12.60%     -1.80%     14.4%
live 21, tanpa BTC  dev 2021-23 risk-on  UTAMA      2785    48.7%      +32.14%    -12.90%     +9.04%     14.3%
                                         PELENGKAP  2785    49.0%      +25.13%    -11.94%     +6.21%     12.9%
                                risk-off UTAMA      2620    42.8%      +16.51%    -13.95%     -0.90%     18.8%
                    uji 2024-26 risk-on  UTAMA      2855    45.1%      +17.69%    -11.87%     +1.46%     12.4%
                                         PELENGKAP  2855    44.4%      +16.98%    -11.80%     +0.97%     12.8%
                                risk-off UTAMA      1637    41.1%       +8.70%    -10.67%     -2.71%     10.0%
live 21, BTC ikut   dev 2021-23 risk-on  UTAMA      2785    48.9%      +31.29%    -12.59%     +8.88%     13.6%
                                         PELENGKAP  2785    48.7%      +24.77%    -11.75%     +6.02%     12.6%
                                risk-off UTAMA      2620    43.1%      +16.08%    -13.75%     -0.90%     18.2%
                    uji 2024-26 risk-on  UTAMA      2855    45.3%      +16.56%    -11.34%     +1.30%     11.3%
                                         PELENGKAP  2855    45.6%      +16.18%    -11.58%     +1.07%     11.9%
                                risk-off UTAMA      1637    41.8%       +8.52%    -10.59%     -2.59%      9.6%

## 2. Konfigurasi live (21 coin, BTC ikut) per tahun

In [5]:
d = SIG["live 21, BTC ikut"]
rows = []
for y in range(2021, 2027):
    p = d[d.year == y]
    for label, m in (("UTAMA risk-on", p.market_ok & (p.group == "UTAMA")),
                     ("PELENGKAP risk-on", p.market_ok & (p.group == "PELENGKAP")),
                     ("UTAMA risk-off", ~p.market_ok & (p.group == "UTAMA"))):
        s = summ(p[m]); rows.append(dict(tahun=y, kelompok=label, n=s["n"], win_rate=s["win_rate"], ekspektasi=s["ekspektasi"]))
t2 = pd.DataFrame(rows).pivot(index="tahun", columns="kelompok", values=["win_rate", "ekspektasi"])
t2.map(lambda v: f"{v:+.2%}" if pd.notna(v) else "-")

win_rate                                     ekspektasi                             
kelompok PELENGKAP risk-on UTAMA risk-off UTAMA risk-on PELENGKAP risk-on UTAMA risk-off UTAMA risk-on
tahun                                                                                                 
2021               +55.00%        +31.21%       +55.74%           +12.08%         -4.87%       +19.87%
2022                +8.46%        +41.30%        +3.85%           -12.55%         -2.15%       -13.72%
2023               +46.90%        +67.18%       +47.25%            +2.54%         +9.99%        +1.59%
2024               +46.45%        +65.66%       +45.50%            +3.07%         +4.97%        +3.39%
2025               +47.98%        +39.87%       +48.54%            -0.33%         -3.77%        -0.23%
2026               +28.00%        +33.99%       +28.44%            -6.25%         -4.41%        -6.05%

## 3. Seberapa sering BTC masuk daftar, dan seberapa sering urutan berubah

In [6]:
d = SIG["live 21, BTC ikut"]
days = d.date.nunique()
btc = d[d.coin == BTC]
print(f"BTC masuk top-10: {btc.date.nunique() / days:.0%} hari | jadi peringkat 1: {(btc['rank'] == 1).sum() / days:.0%} hari")
print("Hasil sinyal BTC (semua pasar):", {k: (f"{v:.1%}" if isinstance(v, float) else v) for k, v in summ(btc).items()})
top1 = d[d["rank"] == 1].set_index("date").coin
print(f"Peringkat 1 berganti coin dari hari sebelumnya: {(top1 != top1.shift()).iloc[1:].mean():.0%} hari")
top5 = d[d["rank"] <= 5].groupby("date").coin.apply(set)
churn = [len(a - b) for a, b in zip(top5.iloc[1:], top5.iloc[:-1])]
print(f"Rata-rata coin baru masuk UTAMA per hari: {np.mean(churn):.1f} dari 5")

BTC masuk top-10: 68% hari | jadi peringkat 1: 7% hari
Hasil sinyal BTC (semua pasar): {'n': 1353, 'win_rate': '52.3%', 'rata2_untung': '9.2%', 'rata2_rugi': '-7.6%', 'ekspektasi': '1.2%', 'kena_stop': '3.8%'}
Peringkat 1 berganti coin dari hari sebelumnya: 39% hari
Rata-rata coin baru masuk UTAMA per hari: 1.2 dari 5


## 4. Angka yang akan ditampilkan API (konfigurasi live, periode uji 2024-01 → 2026-06)

In [7]:
API_STATS = T1.loc["live 21, BTC ikut", "uji 2024-26"].reset_index()
API_STATS["label"] = API_STATS.kelompok + " / " + API_STATS.pasar
API_STATS = API_STATS.set_index("label")[["n", "win_rate", "rata2_untung", "rata2_rugi", "ekspektasi", "kena_stop"]]
print({k: {c: (round(float(v), 4) if c != "n" else int(v)) for c, v in r.items()} for k, r in API_STATS.iterrows()})
fmt(API_STATS)

{'UTAMA / risk-on': {'n': 2855, 'win_rate': 0.4529, 'rata2_untung': 0.1656, 'rata2_rugi': -0.1134, 'ekspektasi': 0.013, 'kena_stop': 0.1135}, 'PELENGKAP / risk-on': {'n': 2855, 'win_rate': 0.4557, 'rata2_untung': 0.1618, 'rata2_rugi': -0.1158, 'ekspektasi': 0.0107, 'kena_stop': 0.1194}, 'UTAMA / risk-off': {'n': 1637, 'win_rate': 0.4184, 'rata2_untung': 0.0852, 'rata2_rugi': -0.1059, 'ekspektasi': -0.0259, 'kena_stop': 0.0959}}


,n,win_rate,rata2_untung,rata2_rugi,ekspektasi,kena_stop
label,,,,,,
UTAMA / risk-on,2855,45.3%,+16.56%,-11.34%,+1.30%,11.3%
PELENGKAP / risk-on,2855,45.6%,+16.18%,-11.58%,+1.07%,11.9%
UTAMA / risk-off,1637,41.8%,+8.52%,-10.59%,-2.59%,9.6%


## Kesimpulan

**0. Paritas kode produksi OK.** Jumlah sinyal identik dengan notebook 05 E2 (2785 / 1565 / 1290); ekspektasi beda
±0,2 poin (dev +9,08 vs +9,14%). Sudah dicek terpisah: **seluruh** 198 hari yang top-5-nya berbeda adalah hari dengan
**skor seri** di batas peringkat 5/6 — produksi memutus seri alfabetis (stabil), notebook lama memakai urutan
`argsort` numpy. Bukan bug. Catatan: skor seri cukup sering (±10% hari) karena skor = rata-rata dua peringkat.

**1. Universe live 21 coin + BTC ikut: kualitas setara riset, sedikit lebih rendah.**
| Uji 2024-01 → 2026-06 | Win rate | Ekspektasi | Kena stop −25% |
|---|---|---|---|
| UTAMA (1–5), risk-on | **45,3%** | **+1,30%** | 11,3% |
| PELENGKAP (6–10), risk-on | 45,6% | +1,07% | 11,9% |
| UTAMA, **risk-off** | 41,8% | **−2,59%** | 9,6% |
| (pembanding) riset 40 coin tanpa BTC, UTAMA risk-on | 45,0% | +1,62% | 16,9% |

Universe lebih kecil → untung rata-rata lebih kecil, tapi kerugian & frekuensi kena stop juga lebih kecil (coin
Large/Mid lebih tenang). Angka di tabel ini yang dipakai API sebagai `statistik_historis`.

**2. BTC ikut ranking tidak merusak.** BTC masuk top-10 di 68% hari, jadi peringkat 1 hanya 7% hari. Sinyal BTC:
win rate 52,3%, ekspektasi +1,2%, jarang kena stop (3,8%). UTAMA dengan/ tanpa BTC: uji +1,30 vs +1,46%, dev +8,88
vs +9,04% — selisih dalam batas kebetulan.

**3. Urutan memang berubah tiap hari** (sesuai keinginan user): peringkat 1 berganti coin di 39% hari; rata-rata
1,2 dari 5 coin UTAMA berganti tiap hari.

**4. ⚠️ Tetap lemah di data terbaru** (konfigurasi live, UTAMA risk-on): 2024 +3,39%, 2025 −0,23%,
**2026 (s/d Juni) −6,05%** (win rate 28%). Risk-off negatif hampir tiap tahun.

### Keputusan implementasi
- Endpoint harian, universe = `flex_params` Large/Mid aktif saat request (minus stablecoin/emas/WBETH), BTC ikut.
- Rencana per coin: beli open besok, pegang 14 hari, stop −25%.
- Statistik historis = tabel di atas; plus label risk-on/off + peringatan; plus riwayat paper trading (dihitung
  ulang dari candle, read-only) — sinyal setelah 30 Juni 2026 adalah out-of-sample murni.
- Kalau `flex_params` berubah banyak (jumlah coin jauh dari 21), statistik ini harus diukur ulang.